In [1]:
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

# credentials loaded from .env, never hardcoded — required by the brief's
# "no hard-coded credentials" security guideline
load_dotenv("../.env")
DATABASE_URL = os.getenv("DATABASE_URL")

engine = create_engine(DATABASE_URL)

# quick connectivity test
with engine.connect() as conn:
    result = conn.execute(text("SELECT version();"))
    print(result.fetchone())

('PostgreSQL 17.6 on x86_64-pc-linux-gnu, compiled by gcc (GCC) 15.2.0, 64-bit',)


In [2]:
print(DATABASE_URL[:30] + "...")

postgresql://postgres.syaxlwav...


In [4]:
from sqlalchemy import text

with engine.connect() as conn:
    conn.execute(text("""
        CREATE TABLE IF NOT EXISTS disruptions (
            disruption_id VARCHAR PRIMARY KEY,
            reason VARCHAR,
            planned BOOLEAN,
            severity VARCHAR,
            overlap_duration_minutes DOUBLE PRECISION,
            overlap_start TIMESTAMP,
            overlap_end TIMESTAMP
        );
    """))
    conn.execute(text("""
        CREATE TABLE IF NOT EXISTS trips_with_disruptions (
            operator_name VARCHAR,
            line_name VARCHAR,
            origin VARCHAR,
            destination VARCHAR,
            trip_date DATE,
            weekday_name VARCHAR,
            departure_time VARCHAR,
            scheduled_datetime TIMESTAMP,
            service_code VARCHAR,
            is_disrupted INTEGER,
            reason VARCHAR,
            planned BOOLEAN,
            severity VARCHAR,
            overlap_duration_minutes DOUBLE PRECISION
        );
    """))
    conn.commit()

print("Tables created.")

Tables created.


In [3]:
import pandas as pd

# loading the processed data via pandas (reading from parquet, not Spark, since
# we're inserting into Postgres and don't need Spark for this step)
trips_pd = pd.read_parquet("../data/processed/trips_with_disruptions")
print("Rows to load:", len(trips_pd))
print(trips_pd.dtypes)

Rows to load: 765089
operator_name                          str
line_name                              str
origin                                 str
destination                            str
trip_date                           object
weekday_name                           str
departure_time                         str
scheduled_datetime          datetime64[ns]
service_code                           str
is_disrupted                         int32
reason                                 str
planned                             object
severity                               str
overlap_duration_minutes           float64
dtype: object


In [7]:
# to_sql handles parameterization internally (no raw string concatenation of values),
# satisfying the brief's SQL injection prevention requirement
trips_pd.to_sql(
    "trips_with_disruptions",
    engine,
    if_exists="append",
    index=False,
    chunksize=5000,
    method="multi"
)
print("Data loaded.")

Data loaded.


In [4]:
with engine.connect() as conn:
    result = conn.execute(text("SELECT COUNT(*) FROM trips_with_disruptions;"))
    print("Row count in database:", result.fetchone()[0])

    sample = conn.execute(text("SELECT * FROM trips_with_disruptions LIMIT 5;"))
    for row in sample:
        print(row)

Row count in database: 1530178
('Arriva Yorkshire', '102', 'Eastmoor', 'Wakefield', datetime.date(2026, 5, 21), 'Thursday', '05:45:00', datetime.datetime(2026, 5, 21, 0, 0), 'PB0000582:3', 1, 'roadworks', True, 'minor', 58739.9833333333)
('Arriva Yorkshire', '102', 'Eastmoor', 'Wakefield', datetime.date(2026, 5, 21), 'Thursday', '06:45:00', datetime.datetime(2026, 5, 21, 1, 0), 'PB0000582:3', 1, 'roadworks', True, 'minor', 58739.9833333333)
('Arriva Yorkshire', '102', 'Eastmoor', 'Wakefield', datetime.date(2026, 5, 21), 'Thursday', '08:45:00', datetime.datetime(2026, 5, 21, 3, 0), 'PB0000582:3', 1, 'roadworks', True, 'minor', 58739.9833333333)
('Arriva Yorkshire', '102', 'Eastmoor', 'Wakefield', datetime.date(2026, 5, 21), 'Thursday', '12:15:00', datetime.datetime(2026, 5, 21, 6, 30), 'PB0000582:3', 1, 'roadworks', True, 'minor', 58739.9833333333)
('Arriva Yorkshire', '102', 'Eastmoor', 'Wakefield', datetime.date(2026, 5, 21), 'Thursday', '13:45:00', datetime.datetime(2026, 5, 21, 8, 0

In [5]:
with engine.connect() as conn:
    result = conn.execute(text("""
        SELECT column_name, data_type 
        FROM information_schema.columns 
        WHERE table_name = 'trips_with_disruptions'
        ORDER BY ordinal_position;
    """))
    for row in result:
        print(row)

('operator_name', 'character varying')
('line_name', 'character varying')
('origin', 'character varying')
('destination', 'character varying')
('trip_date', 'date')
('weekday_name', 'character varying')
('departure_time', 'character varying')
('scheduled_datetime', 'timestamp without time zone')
('service_code', 'character varying')
('is_disrupted', 'integer')
('reason', 'character varying')
('planned', 'boolean')
('severity', 'character varying')
('overlap_duration_minutes', 'double precision')


In [6]:
with engine.connect() as conn:
    conn.execute(text("TRUNCATE TABLE trips_with_disruptions;"))
    conn.commit()

print("Table cleared.")

Table cleared.


In [7]:
trips_pd = pd.read_parquet("../data/processed/trips_with_disruptions")

trips_pd.to_sql(
    "trips_with_disruptions",
    engine,
    if_exists="append",
    index=False,
    chunksize=5000,
    method="multi"
)
print("Data loaded.")

Data loaded.


In [8]:
with engine.connect() as conn:
    result = conn.execute(text("SELECT COUNT(*) FROM trips_with_disruptions;"))
    print("Row count in database:", result.fetchone()[0])

Row count in database: 765089


In [9]:
with engine.connect() as conn:
    result = conn.execute(text("""
        SELECT severity, COUNT(*) as trip_count, AVG(overlap_duration_minutes) as avg_duration
        FROM trips_with_disruptions
        WHERE is_disrupted = 1
        GROUP BY severity
        ORDER BY trip_count DESC;
    """))
    for row in result:
        print(row)

('severe', 135967, 131039.983333415)
('moderate', 38859, 98876.8513175929)
('minor', 6123, 38958.6475502208)
